# Run all final QM640 notebooks

Upload this whole `FinalNotebooks` folder to Google Drive, then run the cells below in Colab. The runner executes notebooks 01 to 10 in order and stops if any notebook fails.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
from datetime import datetime

PROJECT_ROOT = Path('/content/drive/MyDrive/QM640_Food_Affordability')
NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks' / 'FinalNotebooks'
EXECUTED_DIR = PROJECT_ROOT / 'notebooks' / 'executed_final'
REPORT_OUTPUT = PROJECT_ROOT / 'reports' / 'notebook_outputs'
EXTERNAL_REQUIRED = PROJECT_ROOT / 'data' / 'external_required'

EXECUTED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_OUTPUT.mkdir(parents=True, exist_ok=True)
EXTERNAL_REQUIRED.mkdir(parents=True, exist_ok=True)

required_inputs = [
    EXTERNAL_REQUIRED / 'agmarknet_enam_arrivals.csv.gz',
    EXTERNAL_REQUIRED / 'des_state_crop_area_production_yield.csv.gz',
    EXTERNAL_REQUIRED / 'hces_2023_24_segment_aggregates.csv.gz',
    EXTERNAL_REQUIRED / 'labour_bureau_rural_wages_monthly.csv.gz',
]
missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError('Missing required input files:\n' + '\n'.join(missing_inputs))

print('Notebook folder:', NOTEBOOK_DIR)
print('Executed-output folder:', EXECUTED_DIR)
print('Required external files found:', len(required_inputs))

In [ ]:
notebooks = [
    '01_data_acquisition_synopsis_aligned.ipynb',
    '02_data_quality_and_cleaning_synopsis_aligned.ipynb',
    '03_exploratory_analysis_synopsis_aligned.ipynb',
    '04_statistical_analysis_synopsis_aligned.ipynb',
    '05_forecasting_models_synopsis_aligned.ipynb',
    '06_shock_classification_synopsis_aligned.ipynb',
    '07_explainability_synopsis_aligned.ipynb',
    '08_affordability_index_synopsis_aligned.ipynb',
    '09_decision_scenario_analysis_synopsis_aligned.ipynb',
    '10_final_results_synopsis_aligned.ipynb',
]

run_log = []
for notebook in notebooks:
    source_path = NOTEBOOK_DIR / notebook
    executed_path = EXECUTED_DIR / notebook
    if not source_path.exists():
        raise FileNotFoundError(f'Missing notebook: {source_path}')

    print('\n' + '=' * 90)
    print('RUNNING:', notebook)
    print('=' * 90)
    start = datetime.now()
    command = [
        sys.executable, '-m', 'jupyter', 'nbconvert',
        '--to', 'notebook',
        '--execute', str(source_path),
        '--output', str(executed_path),
        '--ExecutePreprocessor.timeout=7200',
        '--ExecutePreprocessor.kernel_name=python3',
    ]
    result = subprocess.run(command, text=True, capture_output=True)
    elapsed = (datetime.now() - start).total_seconds()
    run_log.append({
        'notebook': notebook,
        'returncode': result.returncode,
        'elapsed_seconds': elapsed,
        'stdout_tail': result.stdout[-2000:],
        'stderr_tail': result.stderr[-4000:],
    })
    (REPORT_OUTPUT / '00_final_rerun_log.json').write_text(json.dumps(run_log, indent=2), encoding='utf-8')
    print(result.stdout[-2000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError(f'Notebook failed: {notebook}. See reports/notebook_outputs/00_final_rerun_log.json')
    print(f'COMPLETED: {notebook} in {elapsed/60:.1f} minutes')

print('\nALL FINAL NOTEBOOKS COMPLETED SUCCESSFULLY')
print('Run log:', REPORT_OUTPUT / '00_final_rerun_log.json')